# 08 Bias, Fairness, and Robustness Evaluation

This notebook evaluates the final cardiac arrest risk model across clinically relevant subgroups and stress-tests model performance under several robustness scenarios.

Outputs:

- `reports/subgroup_performance.csv`
- `reports/subgroup_missingness_tests.csv`
- `reports/robustness_checks.csv`

Subgroups checked: age bands, gender, smoking status, alcohol status, family history of cardiovascular disease, GCS severity category, and triage score group.

In [1]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

# Resolve paths robustly whether the notebook is run from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'CardiacPatientData.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import add_clinical_features

DATA_PATH = PROJECT_ROOT / 'data' / 'CardiacPatientData.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'best_model.joblib'
REPORTS_DIR = PROJECT_ROOT / 'reports'
SUBGROUP_PERFORMANCE_PATH = REPORTS_DIR / 'subgroup_performance.csv'
SUBGROUP_MISSINGNESS_PATH = REPORTS_DIR / 'subgroup_missingness_tests.csv'
ROBUSTNESS_PATH = REPORTS_DIR / 'robustness_checks.csv'
FINAL_METRICS_PATH = REPORTS_DIR / 'final_model_metrics.csv'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
POS_LABEL = 1
MIN_CLASS_COUNT_FOR_RANK_METRICS = 2


## Helper functions

The helpers below standardize feature preparation, model training, metrics, subgroup definitions, and missingness comparisons. Metrics that require both outcome classes (AUROC and AUPRC) are reported as missing when a subgroup contains only one class.

In [2]:
RAW_CATEGORICAL_FEATURES = ['Gender', 'Alcoholic', 'Smoke', 'FHCD', 'TriageScore']
ENGINEERED_CATEGORICAL_FEATURES = [
    'age_band',
    'gcs_severity',
    'triage_score_group',
    'hypoxemia_flag',
    'sodium_abnormal_flag',
    'potassium_abnormal_flag',
    'chloride_abnormal_flag',
    'urea_abnormal_flag',
    'creatinine_abnormal_flag',
]
RAW_NUMERIC_FEATURES = ['SBP', 'DBP', 'HR', 'RR', 'BT', 'SpO2', 'Age', 'GCS', 'Na', 'K', 'Cl', 'Urea', 'Ceratinine']


def add_triage_score_group(df):
    '''Create a readable triage score group while preserving missing values.'''
    result = df.copy()
    if 'TriageScore' not in result.columns:
        return result
    triage = result['TriageScore']
    result['triage_score_group'] = np.select(
        [triage.isna(), triage <= 2, triage == 3, triage >= 4],
        ['Missing', '1-2 / highest acuity', '3 / urgent', '4-5 / lower acuity'],
        default='Other / out of range',
    )
    result.loc[triage.isna(), 'triage_score_group'] = np.nan
    return result


def engineer_features(df, include_engineered=True):
    '''Return model features with deterministic clinical features added when requested.'''
    features = df.copy()
    if include_engineered:
        features = add_clinical_features(features)
        features = add_triage_score_group(features)
    return features


def prepare_model_matrix(df, include_engineered=True):
    '''Drop ID and coerce categorical columns to object for consistent one-hot encoding.'''
    X = engineer_features(df, include_engineered=include_engineered).drop(columns=['ID'], errors='ignore')
    candidate_categorical = RAW_CATEGORICAL_FEATURES + (ENGINEERED_CATEGORICAL_FEATURES if include_engineered else [])
    categorical_features = [col for col in candidate_categorical if col in X.columns]
    numeric_features = [col for col in X.columns if col not in categorical_features]
    X_model = X.copy()
    for col in categorical_features:
        X_model[col] = X_model[col].astype('object').where(X_model[col].notna(), np.nan)
    return X_model, numeric_features, categorical_features


def build_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipeline, numeric_features),
            ('categorical', categorical_pipeline, categorical_features),
        ],
        sparse_threshold=0.0,
    )


def build_default_final_model(numeric_features, categorical_features, random_state=RANDOM_STATE):
    '''Fallback final-model specification used when models/best_model.joblib is not present.'''
    return Pipeline([
        ('preprocess', build_preprocessor(numeric_features, categorical_features)),
        ('model', RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=1,
            class_weight=None,
            random_state=random_state,
            n_jobs=-1,
        )),
    ])


def get_holdout_split(df, random_state=RANDOM_STATE):
    y = df['Outcome'].astype(int)
    ids_repeat = bool(df['ID'].duplicated().any())
    if ids_repeat:
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_state)
        train_idx, test_idx = next(splitter.split(df.drop(columns=['Outcome']), y, groups=df['ID']))
        split_strategy = 'StratifiedGroupKFold by patient ID'
    else:
        train_idx, test_idx = train_test_split(
            np.arange(len(df)),
            test_size=TEST_SIZE,
            stratify=y,
            random_state=random_state,
        )
        split_strategy = 'Stratified train/test split'
    return train_idx, test_idx, split_strategy


def fit_or_load_final_model(train_raw, include_engineered=True):
    X_train, numeric_features, categorical_features = prepare_model_matrix(
        train_raw.drop(columns=['Outcome']),
        include_engineered=include_engineered,
    )
    y_train = train_raw['Outcome'].astype(int)
    if MODEL_PATH.exists() and include_engineered:
        model = joblib.load(MODEL_PATH)
        source = str(MODEL_PATH.relative_to(PROJECT_ROOT))
    else:
        model = build_default_final_model(numeric_features, categorical_features, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        source = 'fallback RandomForestClassifier trained in this notebook'
    return model, source


def score_model(model, raw_df, include_engineered=True):
    X, _, _ = prepare_model_matrix(raw_df.drop(columns=['Outcome']), include_engineered=include_engineered)
    y = raw_df['Outcome'].astype(int)
    y_score = model.predict_proba(X)[:, 1]
    return X, y, y_score


def threshold_metrics(y_true, y_score, threshold):
    y_true = pd.Series(y_true).astype(int)
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'precision_ppv': precision_score(y_true, y_pred, zero_division=0),
        'npv': tn / (tn + fn) if (tn + fn) else np.nan,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
    }


def discrimination_metrics(y_true, y_score):
    y_true = pd.Series(y_true).astype(int)
    class_counts = y_true.value_counts()
    if y_true.nunique() < 2 or class_counts.min() < MIN_CLASS_COUNT_FOR_RANK_METRICS:
        return {'auroc': np.nan, 'auprc': np.nan}
    return {
        'auroc': roc_auc_score(y_true, y_score),
        'auprc': average_precision_score(y_true, y_score),
    }


def all_metrics(y_true, y_score, threshold):
    metrics = {
        'sample_size': int(len(y_true)),
        'outcome_rate': float(pd.Series(y_true).mean()) if len(y_true) else np.nan,
    }
    metrics.update(discrimination_metrics(y_true, y_score))
    metrics.update(threshold_metrics(y_true, y_score, threshold))
    return metrics


def safe_chi_square_pvalue(group_values, missing_indicator):
    table = pd.crosstab(group_values.fillna('Missing'), missing_indicator.astype(int))
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    try:
        return float(chi2_contingency(table)[1])
    except ValueError:
        return np.nan


## Load data, recreate the held-out test split, and score the final model

This mirrors prior notebooks by keeping repeated patient IDs together in the hold-out split. If a saved model artifact is available, it is used directly. If not, the notebook trains the same Random Forest final-model fallback used for robustness comparisons so that the bias analysis remains executable from a fresh clone.

In [3]:
df = pd.read_csv(DATA_PATH)
train_idx, test_idx, split_strategy = get_holdout_split(df, random_state=RANDOM_STATE)
train_raw = df.iloc[train_idx].copy()
test_raw = df.iloc[test_idx].copy()

model, model_source = fit_or_load_final_model(train_raw, include_engineered=True)
X_test_model, y_test, y_score = score_model(model, test_raw, include_engineered=True)

if FINAL_METRICS_PATH.exists():
    threshold = float(pd.read_csv(FINAL_METRICS_PATH).loc[0, 'recommended_threshold'])
    threshold_source = str(FINAL_METRICS_PATH.relative_to(PROJECT_ROOT))
else:
    threshold = 0.50
    threshold_source = 'default threshold because saved final metrics/model artifact was not available'

overall_metrics = all_metrics(y_test, y_score, threshold)

print(f'Data: {DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'Model source: {model_source}')
print(f'Split strategy: {split_strategy}')
print(f'Train rows: {len(train_raw):,}; test rows: {len(test_raw):,}')
print(f'Test event rate: {y_test.mean():.3f}')
print(f'Threshold: {threshold:.2f} ({threshold_source})')
print(pd.Series(overall_metrics))


Data: data\CardiacPatientData.csv
Model source: models\best_model.joblib
Split strategy: StratifiedGroupKFold by patient ID
Train rows: 4,826; test rows: 1,080
Test event rate: 0.881
Threshold: 0.50 (default threshold because saved final metrics/model artifact was not available)
sample_size       1080.000000
outcome_rate         0.881481
auroc                0.891442
auprc                0.983477
sensitivity          0.977941
specificity          0.570312
precision_ppv        0.944219
npv                  0.776596
f1                   0.960784
true_negative       73.000000
false_positive      55.000000
false_negative      21.000000
true_positive      931.000000
dtype: float64


## Subgroup performance

For each subgroup level, the table reports sample size, outcome rate, AUROC, AUPRC, sensitivity, specificity, PPV, NPV, and F1. Threshold-dependent metrics use the final-model recommended threshold when available, otherwise 0.50.

In [4]:
subgroup_frame = engineer_features(test_raw.drop(columns=['Outcome']), include_engineered=True)
subgroup_frame['Outcome'] = test_raw['Outcome'].astype(int).values
subgroup_frame['predicted_risk'] = y_score

# Human-readable labels. Keep binary values explicit because the source data dictionary may define coding separately.
subgroup_definitions = {
    'Age bands': 'age_band',
    'Gender': 'Gender',
    'Smoking status': 'Smoke',
    'Alcohol status': 'Alcoholic',
    'Family history of cardiovascular disease': 'FHCD',
    'GCS severity category': 'gcs_severity',
    'Triage score group': 'triage_score_group',
}

rows = []
for subgroup_name, column in subgroup_definitions.items():
    values = subgroup_frame[column].astype('object').where(subgroup_frame[column].notna(), 'Missing')
    for level in sorted(values.unique(), key=lambda x: str(x)):
        mask = values == level
        row = {
            'subgroup': subgroup_name,
            'subgroup_variable': column,
            'subgroup_level': str(level),
        }
        row.update(all_metrics(subgroup_frame.loc[mask, 'Outcome'], subgroup_frame.loc[mask, 'predicted_risk'], threshold))
        rows.append(row)

subgroup_performance = pd.DataFrame(rows)
metric_order = [
    'subgroup', 'subgroup_variable', 'subgroup_level', 'sample_size', 'outcome_rate',
    'auroc', 'auprc', 'sensitivity', 'specificity', 'precision_ppv', 'npv', 'f1',
    'true_negative', 'false_positive', 'false_negative', 'true_positive',
]
subgroup_performance = subgroup_performance[metric_order]
subgroup_performance.to_csv(SUBGROUP_PERFORMANCE_PATH, index=False)

print(f'Saved subgroup performance to: {SUBGROUP_PERFORMANCE_PATH.relative_to(PROJECT_ROOT)}')
display(subgroup_performance)


Saved subgroup performance to: reports\subgroup_performance.csv


,subgroup,subgroup_variable,subgroup_level,sample_size,outcome_rate,auroc,auprc,sensitivity,specificity,precision_ppv,npv,f1,true_negative,false_positive,false_negative,true_positive
0,Age bands,age_band,40-64,454,1.000000,NaN,NaN,0.991189,NaN,1.000000,0.000000,0.995575,0,0,4,450
1,Age bands,age_band,65-79,452,0.716814,0.936282,0.976260,0.962963,0.570312,0.850136,0.858824,0.903039,73,55,12,312
2,Age bands,age_band,80+,68,1.000000,NaN,NaN,0.926471,NaN,1.000000,0.000000,0.961832,0,0,5,63
3,Age bands,age_band,<18,106,1.000000,NaN,NaN,1.000000,NaN,1.000000,NaN,1.000000,0,0,0,106
4,Gender,Gender,0,237,1.000000,NaN,NaN,0.953586,NaN,1.000000,0.000000,0.976242,0,0,11,226
5,Gender,Gender,1,843,0.848161,0.885140,0.976717,0.986014,0.570312,0.927632,0.879518,0.955932,73,55,10,705
6,Smoking status,Smoke,0.0,627,0.795853,0.930376,0.981069,0.991984,0.570312,0.900000,0.948052,0.943756,73,55,4,495
7,Smoking status,Smoke,1.0,165,1.000000,NaN,NaN,1.000000,NaN,1.000000,NaN,1.000000,0,0,0,165
8,Smoking status,Smoke,Missing,288,1.000000,NaN,NaN,0.940972,NaN,1.000000,0.000000,0.969589,0,0,17,271
9,Alcohol status,Alcoholic,0.0,575,0.871304,0.994120,0.999238,0.992016,0.986486,0.997992,0.948052,0.994995,73,1,4,497


## Missingness by subgroup

The table below evaluates whether predictor missingness varies across each subgroup definition. The `any_predictor_missing_rate` values are subgroup-level descriptive rates. The chi-square tests screen each raw predictor's missingness indicator against the subgroup variable, then report the minimum p-value and variables with p < 0.05. These tests are exploratory and unadjusted for multiple comparisons.

In [5]:
missingness_base = engineer_features(test_raw.drop(columns=['Outcome']), include_engineered=True)
raw_predictor_cols = [col for col in df.columns if col not in ['ID', 'Outcome']]
missingness_base['any_predictor_missing'] = missingness_base[raw_predictor_cols].isna().any(axis=1)

missingness_rows = []
for subgroup_name, column in subgroup_definitions.items():
    group_values = missingness_base[column].astype('object').where(missingness_base[column].notna(), 'Missing')
    pvalue_rows = []
    for predictor in raw_predictor_cols:
        pvalue_rows.append({
            'predictor': predictor,
            'missingness_p_value': safe_chi_square_pvalue(group_values, missingness_base[predictor].isna()),
        })
    pvalues = pd.DataFrame(pvalue_rows).dropna(subset=['missingness_p_value'])
    significant_predictors = pvalues.loc[pvalues['missingness_p_value'] < 0.05, 'predictor'].tolist()
    min_p = pvalues['missingness_p_value'].min() if len(pvalues) else np.nan

    for level in sorted(group_values.unique(), key=lambda x: str(x)):
        mask = group_values == level
        missingness_rows.append({
            'subgroup': subgroup_name,
            'subgroup_variable': column,
            'subgroup_level': str(level),
            'sample_size': int(mask.sum()),
            'any_predictor_missing_count': int(missingness_base.loc[mask, 'any_predictor_missing'].sum()),
            'any_predictor_missing_rate': float(missingness_base.loc[mask, 'any_predictor_missing'].mean()),
            'minimum_predictor_missingness_p_value': min_p,
            'n_predictors_with_missingness_p_lt_0_05': len(significant_predictors),
            'predictors_with_missingness_p_lt_0_05': ', '.join(significant_predictors),
        })

subgroup_missingness = pd.DataFrame(missingness_rows)
subgroup_missingness.to_csv(SUBGROUP_MISSINGNESS_PATH, index=False)

# Add missingness descriptors to the primary subgroup performance report as requested.
subgroup_performance_with_missingness = subgroup_performance.merge(
    subgroup_missingness,
    on=['subgroup', 'subgroup_variable', 'subgroup_level', 'sample_size'],
    how='left',
)
subgroup_performance_with_missingness.to_csv(SUBGROUP_PERFORMANCE_PATH, index=False)

print(f'Saved missingness tests to: {SUBGROUP_MISSINGNESS_PATH.relative_to(PROJECT_ROOT)}')
print(f'Updated subgroup performance with missingness columns: {SUBGROUP_PERFORMANCE_PATH.relative_to(PROJECT_ROOT)}')
display(subgroup_missingness)


Saved missingness tests to: reports\subgroup_missingness_tests.csv
Updated subgroup performance with missingness columns: reports\subgroup_performance.csv


,subgroup,subgroup_variable,subgroup_level,sample_size,any_predictor_missing_count,any_predictor_missing_rate,minimum_predictor_missingness_p_value,n_predictors_with_missingness_p_lt_0_05,predictors_with_missingness_p_lt_0_05
0,Age bands,age_band,40-64,454,275,0.605727,1.138580e-13,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
1,Age bands,age_band,65-79,452,267,0.590708,1.138580e-13,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
2,Age bands,age_band,80+,68,38,0.558824,1.138580e-13,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
3,Age bands,age_band,<18,106,39,0.367925,1.138580e-13,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
4,Gender,Gender,0,237,163,0.687764,3.126481e-61,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
5,Gender,Gender,1,843,456,0.540925,3.126481e-61,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
6,Smoking status,Smoke,0.0,627,241,0.384370,3.026772e-235,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
7,Smoking status,Smoke,1.0,165,90,0.545455,3.026772e-235,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
8,Smoking status,Smoke,Missing,288,288,1.000000,3.026772e-235,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
9,Alcohol status,Alcoholic,0.0,575,188,0.326957,3.026772e-235,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."


## Robustness checks

Robustness checks rerun a comparable Random Forest modeling pipeline under:

1. Alternative leakage-safe train/test splits.
2. Outlier handling through train-set IQR removal and train-derived winsorization.
3. Feature-set comparison with and without engineered clinical features.

These are not intended to replace the final model selection workflow; they identify whether conclusions are sensitive to reasonable analysis choices.

In [6]:
def fit_evaluate_pipeline(train_df, test_df, scenario, random_state=RANDOM_STATE, include_engineered=True):
    X_train, numeric_features, categorical_features = prepare_model_matrix(
        train_df.drop(columns=['Outcome']),
        include_engineered=include_engineered,
    )
    X_test, _, _ = prepare_model_matrix(
        test_df.drop(columns=['Outcome']),
        include_engineered=include_engineered,
    )
    y_train = train_df['Outcome'].astype(int)
    y_eval = test_df['Outcome'].astype(int)
    estimator = build_default_final_model(numeric_features, categorical_features, random_state=random_state)
    estimator.fit(X_train, y_train)
    eval_score = estimator.predict_proba(X_test)[:, 1]
    row = {
        'scenario': scenario,
        'random_state': random_state,
        'include_engineered_features': include_engineered,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'test_event_rate': float(y_eval.mean()),
        'threshold': threshold,
    }
    row.update(all_metrics(y_eval, eval_score, threshold))
    return row


def remove_train_outliers_iqr(train_df, numeric_cols, multiplier=1.5):
    train_clean = train_df.copy()
    mask = pd.Series(True, index=train_clean.index)
    for col in numeric_cols:
        if col not in train_clean.columns:
            continue
        q1 = train_clean[col].quantile(0.25)
        q3 = train_clean[col].quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr
        mask &= train_clean[col].isna() | train_clean[col].between(lower, upper)
    return train_clean.loc[mask].copy()


def winsorize_from_train(train_df, test_df, numeric_cols, lower_q=0.01, upper_q=0.99):
    train_w = train_df.copy()
    test_w = test_df.copy()
    for col in numeric_cols:
        if col not in train_w.columns:
            continue
        lower = train_w[col].quantile(lower_q)
        upper = train_w[col].quantile(upper_q)
        train_w[col] = train_w[col].clip(lower, upper)
        test_w[col] = test_w[col].clip(lower, upper)
    return train_w, test_w

robustness_rows = []

# Baseline comparable pipeline on the primary split.
robustness_rows.append(fit_evaluate_pipeline(train_raw, test_raw, 'primary split / engineered features', RANDOM_STATE, True))

# Alternative splits.
for seed in [7, 2024, 2026]:
    alt_train_idx, alt_test_idx, _ = get_holdout_split(df, random_state=seed)
    alt_train = df.iloc[alt_train_idx].copy()
    alt_test = df.iloc[alt_test_idx].copy()
    robustness_rows.append(fit_evaluate_pipeline(alt_train, alt_test, f'alternative split seed {seed}', seed, True))

# Outlier removal / winsorization on the primary split.
outlier_removed_train = remove_train_outliers_iqr(train_raw, RAW_NUMERIC_FEATURES, multiplier=1.5)
robustness_rows.append(fit_evaluate_pipeline(outlier_removed_train, test_raw, 'train IQR outlier removal', RANDOM_STATE, True))

winsor_train, winsor_test = winsorize_from_train(train_raw, test_raw, RAW_NUMERIC_FEATURES, lower_q=0.01, upper_q=0.99)
robustness_rows.append(fit_evaluate_pipeline(winsor_train, winsor_test, 'train-derived 1st/99th percentile winsorization', RANDOM_STATE, True))

# With vs without engineered features.
robustness_rows.append(fit_evaluate_pipeline(train_raw, test_raw, 'primary split / raw features only', RANDOM_STATE, False))

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(ROBUSTNESS_PATH, index=False)

print(f'Saved robustness checks to: {ROBUSTNESS_PATH.relative_to(PROJECT_ROOT)}')
display(robustness_df)


Saved robustness checks to: reports\robustness_checks.csv


,scenario,random_state,include_engineered_features,train_rows,test_rows,test_event_rate,threshold,sample_size,outcome_rate,auroc,auprc,sensitivity,specificity,precision_ppv,npv,f1,true_negative,false_positive,false_negative,true_positive
0,primary split / engineered features,42,True,4826,1080,0.881481,0.5,1080,0.881481,0.837969,0.969623,0.934874,0.562500,0.940803,0.537313,0.937829,72,56,62,890
1,alternative split seed 7,7,True,4751,1155,0.859740,0.5,1155,0.859740,0.832566,0.966074,0.946626,0.475309,0.917073,0.592308,0.931615,77,85,53,940
2,alternative split seed 2024,2024,True,4847,1059,0.890463,0.5,1059,0.890463,0.796372,0.969908,0.976670,0.181034,0.906496,0.488372,0.940276,21,95,22,921
3,alternative split seed 2026,2026,True,4868,1038,0.731214,0.5,1038,0.731214,0.892723,0.953629,0.989460,0.200717,0.771047,0.875000,0.866705,56,223,8,751
4,train IQR outlier removal,42,True,3602,1080,0.881481,0.5,1080,0.881481,0.796317,0.962365,0.936975,0.562500,0.940928,0.545455,0.938947,72,56,60,892
5,train-derived 1st/99th percentile winsorization,42,True,4826,1080,0.881481,0.5,1080,0.881481,0.842741,0.972573,0.933824,0.562500,0.940741,0.533333,0.937269,72,56,63,889
6,primary split / raw features only,42,False,4826,1080,0.881481,0.5,1080,0.881481,0.872518,0.977814,0.934874,0.562500,0.940803,0.537313,0.937829,72,56,62,890


## Bias, fairness, and robustness discussion

### How to interpret subgroup results

- **Small subgroup sizes:** AUROC and AUPRC are unstable when a subgroup has few records or very few non-events/events. This notebook leaves AUROC/AUPRC blank when a subgroup has only one outcome class.
- **Outcome-rate differences:** Large differences in outcome rate can change PPV and NPV even if AUROC is similar. PPV generally increases and NPV generally decreases in higher-prevalence subgroups.
- **Threshold effects:** Sensitivity, specificity, PPV, NPV, and F1 are evaluated at a single global threshold. A global threshold may not yield equal operating characteristics across subgroups if baseline risk, measurement patterns, or calibration differ.
- **Missingness:** Missing data can encode care processes, documentation quality, or disease severity. If missingness differs by subgroup, performance gaps may partly reflect how tests and vital signs were collected rather than true clinical risk.

### Possible bias sources

- **Selection bias:** The dataset may include only patients captured by this study's inclusion criteria, hospital workflows, or emergency-care documentation systems. Patients treated elsewhere, transferred, discharged quickly, or missing key records may be underrepresented.
- **Measurement bias:** Vitals, laboratory values, triage scores, smoking/alcohol history, and GCS can be measured or documented differently across clinicians, time periods, and patient groups. Differential measurement frequency can also produce subgroup-specific missingness.
- **Repeated patient records:** Repeated `ID` values suggest multiple rows may come from the same patient. The split keeps IDs together, but subgroup estimates can still be influenced by high-utilization patients contributing multiple observations.
- **Single-site data:** If the data come from one site or health system, the model may learn local practice patterns, triage norms, patient demographics, or equipment/documentation conventions that do not generalize to other settings.
- **Label quality:** `Outcome` may be affected by coding accuracy, timing of cardiac arrest ascertainment, censoring, delayed events, or ambiguous clinical definitions. Label noise can reduce apparent performance and may be worse for some subgroups.

### Practical recommendations

- Review subgroups with low sensitivity or low NPV before clinical use, because missed high-risk patients are the main safety concern for screening.
- Validate externally on data from additional sites and time periods.
- Consider patient-level analysis if repeated records reflect longitudinal measurements, including a sensitivity analysis that samples one record per patient.
- Audit missingness mechanisms and documentation practices, especially for predictors with significant subgroup missingness differences.
- Assess calibration by subgroup before using risk probabilities for decision thresholds or resource allocation.

# Final Conclusion

This notebook shows that the cardiac arrest risk model performs well overall, but its performance is not evenly reliable across every subgroup. The model had strong test-set performance, with high sensitivity and precision, making it useful as a potential screening or prioritization tool. However, the lower specificity means it still produces a notable number of false positives, so the model should be interpreted as a risk flag rather than a final clinical decision-maker. The fairness and subgroup results are the most important part of this analysis. Some groups showed strong performance, while others had unstable or concerning results, especially where outcome rates were extremely high or where subgroup sizes were limited. Missingness was also not evenly distributed across groups, suggesting that the model may be learning patterns from clinical documentation, testing behavior, or hospital workflow in addition to true patient risk. This matters because a model can look strong overall while still being less reliable for specific patient populations.

Overall, the model is a promising proof of concept, but it needs more validation before being treated as clinically actionable. Future work should focus on external validation, calibration checks, subgroup-specific error analysis, and a closer audit of missing data patterns. Also, hidden note for anyone actually reading through this notebook: Joo Sae hyuk is my favorite table tennis player, which is unrelated to cardiac arrest modeling but absolutely relevant to my personal interests.
